# Лекция 2. Основы логики и язык OWL: демонстрация

Курс «От интеллектуального поиска к интеллектуальным агентам», ВМК МГУ. О. М. Атаева, А. П. Халов.

Блокнот продолжает задачу редактора из лекции 1. Положение о статьях, журнал рецензий и статья P2 записываются
как онтология. Машина вывода получает факты, которых никто не записывал, и обнаруживает противоречие в данных.

| Шаг | Что делаем |
|---|---|
| 0 | Загружаем данные лекции 1 |
| 1 | Классы и дизъюнктность |
| 2 | Свойства и их характеристики |
| 3 | Индивиды из корпуса и журнала |
| 4 | Аксиомы и правило для требования 6.2 |
| 5 | Сохранение, просмотр в Protégé |
| 6 | Запуск машины вывода: четыре вывода |
| 7 | Те же выводы своими руками: прямой вывод по правилам и три матричные операции |
| 8 | Строка журнала с ошибкой: противоречие |
| 9 | Чего в выводе нет |
| 10 | Агент с онтологией |
| 11 | Вариант по математике |

Для шагов 6 и 8 нужна Java. Без неё блокнот печатает сохранённую запись вывода, шаг 7 работает всегда.

## Шаг 0. Данные лекции 1

In [1]:
import pathlib, datetime, warnings, io, contextlib, os, json, re, shutil
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")
from owlready2 import *

DATA = pathlib.Path("data")
corpus  = pd.read_csv(DATA / "corpus_Aprime_standards.csv", dtype=str).fillna("")
reviews = pd.read_csv(DATA / "reviews.csv", dtype=str).fillna("")
claims  = pd.read_csv(DATA / "paper_P2_claims.csv", dtype=str).fillna("")
HAS_JAVA = shutil.which("java") is not None
print(len(corpus), "положение;", len(reviews), "строк журнала;", len(claims), "утверждения статьи P2; Java:", HAS_JAVA)
corpus[corpus["id"].isin(["S06", "T06", "R62", "R64", "R65"])][["id", "тип", "название", "зависит_от"]]

31 положение; 14 строк журнала; 3 утверждения статьи P2; Java: True


,id,тип,название,зависит_от
8,T06,Термин,Рецензия,S03;T05
19,S06,Раздел,Раздел 6. Порядок рецензирования,S01;S03
21,R62,Требование,6.2 Независимость рецензента,S06;T05;T04
23,R64,Требование,6.4 Ответ на замечания,S06;R63;T08
24,R65,Требование,6.5 Принятие решения,S06;R61;R64;T10


## Модель: настройка

Живые ячейки шага 10 (агент с онтологией) обращаются к языковой модели через OpenAI-совместимый API. Три значения
задаются здесь явно: ключ, адрес сервера и имя модели. Ячейка проверяет, отвечает ли сервер. Если нет, ключ
сбрасывается, и шаг 10 печатает записанный протокол, остальные шаги от модели не зависят.

In [2]:
import os

# Вариант 1. Локальная Ollama: сервер на этой машине, модель с вызовом функций (qwen-agent собрана из Modelfile
#            командой ollama create qwen-agent -f Modelfile, подойдёт и qwen2.5:3b). Ключ для Ollama любая непустая строка.
LLM_API_KEY  = "ollama"
LLM_BASE_URL = "http://127.0.0.1:11434/v1"
LLM_MODEL    = "qwen-agent"

# Вариант 2. Облачная модель: подставить свои значения вместо трёх строк выше, например
#   LLM_API_KEY = "sk-...";  LLM_BASE_URL = "https://api.openai.com/v1";  LLM_MODEL = "gpt-4o-mini"
# Вариант 3. Без модели: LLM_API_KEY = "". Шаг 10 печатает записанный протокол.

os.environ["LLM_BASE_URL"], os.environ["LLM_MODEL"] = LLM_BASE_URL, LLM_MODEL
if LLM_BASE_URL.startswith(("http://127.0.0.1", "http://localhost")):
    for k in ("ALL_PROXY", "all_proxy", "HTTP_PROXY", "http_proxy", "HTTPS_PROXY", "https_proxy"):
        os.environ.pop(k, None)    # локальный сервер: системный прокси этому блокноту не нужен

# Проверка сервера. Ключ попадает в окружение только если сервер ответил.
os.environ["LLM_API_KEY"] = ""
if LLM_API_KEY:
    try:
        from openai import OpenAI
    except ModuleNotFoundError:
        OpenAI = None
        print("пакет openai не установлен в ядре блокнота: выполните %pip install openai и перезапустите ядро. Живые ячейки идут по записи")
    try:
        if OpenAI is None:
            raise RuntimeError("нет пакета openai")
        names = [m.id for m in OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL, timeout=5, max_retries=0).models.list()]
        os.environ["LLM_API_KEY"] = LLM_API_KEY
        if names and LLM_MODEL not in names and LLM_MODEL + ":latest" not in names:
            print(f"сервер отвечает, модели {LLM_MODEL} в его списке нет (есть: {', '.join(names[:6])}), вызовы могут не пройти")
        else:
            print(f"модель: {LLM_MODEL} через {LLM_BASE_URL}")
    except Exception as e:
        if OpenAI is not None:
            print(f"сервер {LLM_BASE_URL} не отвечает ({type(e).__name__}): шаг 10 идёт по записи")
else:
    print("модель не задана: шаг 10 идёт по записи")

пакет openai не установлен в ядре блокнота: выполните %pip install openai и перезапустите ядро. Живые ячейки идут по записи


## Шаг 1. Классы и дизъюнктность

Карта знаний лекции 1 переносится почти без изменений. Класс в Owlready2 записывается как класс Python.
`AllDisjoint` объявляет классы попарно непересекающимися.

In [3]:
world = World()
onto = world.get_ontology("http://example.org/review.owl")

with onto:
    class Объект(Thing): pass
    class Положение(Объект): pass
    class Раздел(Положение): pass
    class Термин(Положение): pass
    class Требование(Положение): pass
    class Статья(Объект): pass
    class Рецензия(Объект): pass
    class Рецензент(Объект): pass
    class Организация(Объект): pass
    class Утверждение(Объект): pass
    class Доказательство(Объект): pass

    AllDisjoint([Раздел, Термин, Требование])
    AllDisjoint([Положение, Статья, Рецензия, Рецензент, Организация, Утверждение, Доказательство])

for c in onto.classes():
    print(f"{c.name:15} ⊑ {', '.join(p.name for p in c.is_a)}")

Объект          ⊑ Thing
Положение       ⊑ Объект
Раздел          ⊑ Положение
Термин          ⊑ Положение
Требование      ⊑ Положение
Статья          ⊑ Объект
Рецензия        ⊑ Объект
Рецензент       ⊑ Объект
Организация     ⊑ Объект
Утверждение     ⊑ Объект
Доказательство  ⊑ Объект


## Шаг 2. Свойства и их характеристики

Объектное свойство связывает два индивида, свойство данных связывает индивид со значением.
Транзитивность `зависит_от` задаётся одним словом `TransitiveProperty`.

In [4]:
with onto:
    class зависит_от(ObjectProperty, TransitiveProperty):
        domain = [Положение]; range = [Положение]
    class на_статью(ObjectProperty, FunctionalProperty):
        domain = [Рецензия]; range = [Статья]
    class имеет_рецензию(ObjectProperty):
        inverse_property = на_статью
    class рецензент(ObjectProperty, FunctionalProperty):
        domain = [Рецензия]; range = [Рецензент]
    class из_организации(ObjectProperty):
        domain = [Рецензент]; range = [Организация]
    class организация_авторов(ObjectProperty):
        domain = [Статья]; range = [Организация]
    class нарушает(ObjectProperty):
        domain = [Статья]; range = [Требование]
    class содержит_утверждение(ObjectProperty):
        domain = [Статья]; range = [Утверждение]
    class доказывается(ObjectProperty):
        domain = [Утверждение]; range = [Доказательство]
    class доказывает(ObjectProperty):
        inverse_property = доказывается

    class оценка(DataProperty, FunctionalProperty):
        domain = [Рецензия]; range = [float]
    class шкала(DataProperty, FunctionalProperty):
        domain = [Рецензия]; range = [int]
    class назначено(DataProperty, FunctionalProperty):
        domain = [Рецензия]; range = [datetime.date]
    class сдано(DataProperty, FunctionalProperty):
        domain = [Рецензия]; range = [datetime.date]
    class ответ_авторов(DataProperty, FunctionalProperty):
        domain = [Рецензия]; range = [bool]
    class вердикт(DataProperty, FunctionalProperty):
        domain = [Доказательство]; range = [str]
    class источник(DataProperty):
        domain = [Положение]; range = [str]

for p in onto.object_properties():
    kinds = [k.__name__.replace("Property", "") for k in p.is_a if k is not ObjectProperty]
    d = p.domain[0].name if p.domain else "?"; r = p.range[0].name if p.range else "?"
    print(f"{p.name:22} {d:12} → {r:14} {', '.join(kinds)}")

зависит_от             Положение    → Положение      Transitive
на_статью              Рецензия     → Статья         Functional
имеет_рецензию         ?            → ?              
рецензент              Рецензия     → Рецензент      Functional
из_организации         Рецензент    → Организация    
организация_авторов    Статья       → Организация    
нарушает               Статья       → Требование     
содержит_утверждение   Статья       → Утверждение    
доказывается           Утверждение  → Доказательство 
доказывает             ?            → ?              


## Шаг 3. Индивиды из корпуса и журнала

Данные те же, что в лекции 1. Каждая запись корпуса становится индивидом класса Раздел, Термин или Требование.
Каждая строка журнала даёт рецензию, статью, рецензента и две организации. Оценка и шкала записываются
двумя отдельными свойствами.

In [5]:
TYPES = {"Раздел": Раздел, "Термин": Термин, "Требование": Требование}
ident = lambda s: s.replace(" ", "_")
day = lambda s: datetime.date.fromisoformat(s)

with onto:
    pos = {}
    for _, r in corpus.iterrows():
        x = TYPES[r["тип"]](r["id"])
        x.label = [locstr(r["название"], lang="ru")]
        x.источник = [r["источник"]]
        pos[r["id"]] = x
    for _, r in corpus.iterrows():
        pos[r["id"]].зависит_от = [pos[d] for d in r["зависит_от"].split(";") if d]

    org, art, rev, recs = {}, {}, {}, []
    for _, r in reviews.iterrows():
        for o in (r["авторы_организация"], r["рецензент_организация"]):
            if o not in org:
                org[o] = Организация(ident(o))
        if r["статья"] not in art:
            art[r["статья"]] = Статья(r["статья"])
            art[r["статья"]].организация_авторов = [org[r["авторы_организация"]]]
        if r["рецензент"] not in rev:
            rev[r["рецензент"]] = Рецензент(r["рецензент"])
            rev[r["рецензент"]].из_организации = [org[r["рецензент_организация"]]]
        x = Рецензия(f"рец_{r['статья']}_{r['рецензент']}")
        x.на_статью = art[r["статья"]]
        x.рецензент = rev[r["рецензент"]]
        x.назначено = day(r["назначено"])
        if r["сдано"]:
            x.сдано = day(r["сдано"]); x.оценка = float(r["оценка"]); x.шкала = int(r["шкала"])
            x.ответ_авторов = (r["ответ_авторов"] == "да")
        recs.append(x)

    for _, c in claims.iterrows():
        u = Утверждение(c["id"]); u.label = [locstr(c["формулировка"], lang="ru")]
        d = Доказательство("Док_" + c["id"]); d.вердикт = c["вердикт_lean"].split(":")[0]
        u.доказывается = [d]
        art["P2"].содержит_утверждение.append(u)

    # термин положения и класс онтологии: слово, понятие, экземпляры
    Рецензия.isDefinedBy = [pos["T06"]]

R62, R65 = pos["R62"], pos["R65"]
print({c.name: len(list(c.instances())) for c in (Раздел, Термин, Требование, Статья, Рецензия, Рецензент, Организация, Утверждение, Доказательство)})
print("R65 зависит_от (записано):", sorted(x.name for x in R65.зависит_от))
x = onto["рец_P2_Rv3"]; print(x.name, "| оценка", x.оценка, "| шкала", x.шкала, "| рецензент", x.рецензент.name, "| статья", x.на_статью.name)

{'Раздел': 6, 'Термин': 11, 'Требование': 14, 'Статья': 8, 'Рецензия': 14, 'Рецензент': 6, 'Организация': 4, 'Утверждение': 3, 'Доказательство': 3}
R65 зависит_от (записано): ['R61', 'R64', 'S06', 'T10']
рец_P2_Rv3 | оценка 4.0 | шкала 5 | рецензент Rv3 | статья P2


## Шаг 4. Аксиомы и правило для требования 6.2

Ограничения записываются добавлением в `is_a`. Определяемый класс задаётся через `equivalent_to`.
Требование 6.2 в лекции 1 было строкой кода. Здесь оно записано правилом в самой онтологии:
пять условий и одно заключение.

In [6]:
with onto:
    Рецензия.is_a.append(на_статью.some(Статья))            # у каждой рецензии есть статья
    Рецензия.is_a.append(рецензент.some(Рецензент))          # и рецензент
    Требование.is_a.append(зависит_от.some(Раздел))          # каждое требование относится к разделу
    Утверждение.is_a.append(доказывается.some(Доказательство))  # у каждого утверждения есть доказательство

    class БазовоеПоложение(Положение):
        equivalent_to = [Положение & Not(зависит_от.some(Положение))]

    правило_62 = Imp()
    правило_62.set_as_rule(
        "Рецензия(?r), на_статью(?r, ?s), рецензент(?r, ?p), из_организации(?p, ?o), организация_авторов(?s, ?o)"
        " -> нарушает(?s, R62)")

for c in (Рецензия, Требование, Утверждение):
    print(c.name, "⊑", " и ".join(str(x).replace("review.", "") for x in c.is_a))
print("БазовоеПоложение ≡", str(БазовоеПоложение.equivalent_to[0]).replace("review.", ""))
print("Правило 6.2:", str(правило_62).replace("review.", ""))

Рецензия ⊑ Объект и на_статью.some(Статья) и рецензент.some(Рецензент)
Требование ⊑ Положение и зависит_от.some(Раздел)
Утверждение ⊑ Объект и доказывается.some(Доказательство)
БазовоеПоложение ≡ Положение & Not(зависит_от.some(Положение))
Правило 6.2: Рецензия(?r), на_статью(?r, ?s), рецензент(?r, ?p), из_организации(?p, ?o), организация_авторов(?s, ?o) -> нарушает(?s, R62)


## Шаг 5. Сохранение и просмотр в Protégé

Онтология сохраняется в двух форматах. Файл `review.owl` открывается в Protégé: вкладка Classes показывает
иерархию, Object properties показывает свойства и их характеристики, Individuals показывает индивидов,
вкладка SWRL показывает правило. Файл `review.ttl` удобен для чтения глазами.

In [7]:
import rdflib
onto.save(file=str(DATA / "review.owl"), format="rdfxml")
g = rdflib.Graph(); g.parse(str(DATA / "review.owl"), format="xml")
g.bind("", "http://example.org/review.owl#")
g.serialize(destination=str(DATA / "review.ttl"), format="turtle")
print(len(g), "троек записано в data/review.owl и data/review.ttl")
ttl = (DATA / "review.ttl").read_text(encoding="utf-8")
print(re.search(r"\n:R65 .*?\.\n", ttl, flags=re.S).group(0))

565 троек записано в data/review.owl и data/review.ttl

:R65 a :Требование,
        owl:NamedIndividual ;
    rdfs:label "6.5 Принятие решения"@ru ;
    :зависит_от :R61,
        :R64,
        :S06,
        :T10 ;
    :источник "Положение о статьях §6.5"^^xsd:string .



## Шаг 6. Запуск машины вывода

Машина вывода Pellet получает онтологию и возвращает факты, которые следуют из записанного.
Смотрим четыре вывода.

In [8]:
def run_reasoner(explain=False):
    """Запускает Pellet. Возвращает (успех, текст сообщения)."""
    if not HAS_JAVA:
        return None, "Java не найдена"
    buf = io.StringIO()
    try:
        with contextlib.redirect_stderr(buf), contextlib.redirect_stdout(buf):
            with onto:
                sync_reasoner_pellet(world, infer_property_values=True, debug=2 if explain else 0)
        return True, ""
    except OwlReadyInconsistentOntologyError as e:
        text = str(e)
        return False, text[text.find("Explanation"):] if "Explanation" in text else text

ok, msg = run_reasoner()
RECORD_6 = '''[запись] вывод машины без Java
1. R65 зависит_от: 14 положений
2. нарушают R62: P3
3. типы R65: Требование, Положение, Объект
4. БазовоеПоложение: до замыкания пусто'''
if ok is None:
    print(RECORD_6)
else:
    deps = sorted(R65.зависит_от, key=lambda x: x.name)
    print("1. R65 зависит_от:", len(deps), "положений")
    print("   ", ", ".join(f"{x.name} {x.label[0]}" for x in deps))
    print("2. нарушают R62:", ", ".join(sorted(a.name for a in art.values() if R62 in a.нарушает)))
    print("3. типы R65:", ", ".join(c.name for c in R65.INDIRECT_is_a if hasattr(c, "name") and c is not Thing))
    print("4. БазовоеПоложение до замыкания:", [x.name for x in БазовоеПоложение.instances()])

1. R65 зависит_от: 14 положений
    R61 6.1 Число рецензий, R63 6.3 Содержание рецензии, R64 6.4 Ответ на замечания, S01 Раздел 1. Область применения, S03 Раздел 3. Термины и определения, S06 Раздел 6. Порядок рецензирования, T01 Статья, T04 Автор, T05 Рецензент, T06 Рецензия, T07 Замечание, T08 Ответ авторов, T09 Редакционная коллегия, T10 Решение
2. нарушают R62: P3
3. типы R65: Требование, Положение, Объект
4. БазовоеПоложение до замыкания: []


Первые три вывода получены без единой строки кода с правилами: транзитивность, правило 6.2 и иерархия классов
записаны в онтологии. Четвёртый вывод пуст. В открытом мире отсутствие записи о зависимости раздела 1
не означает, что зависимости нет. Сообщим машине, что список зависимостей у разделов 1 и 2 полон.

In [9]:
if HAS_JAVA:
    with onto:
        close_world(pos["S01"], Properties=[зависит_от])
        close_world(pos["S02"], Properties=[зависит_от])
    run_reasoner()
    print("4. БазовоеПоложение после замыкания:", sorted(x.name for x in БазовоеПоложение.instances()))
else:
    print("[запись] 4. БазовоеПоложение после замыкания: ['S01', 'S02']")

4. БазовоеПоложение после замыкания: ['S01', 'S02']


## Шаг 7. Те же выводы своими руками

### 7а. Прямой вывод по правилам Хорна

Факты берём из тех же CSV как тройки. Правило Хорна: несколько условий и одно заключение.
Правила применяются к фактам, новые факты добавляются, шаг повторяется до неподвижной точки.
Машина вывода для этого шага не нужна.

In [10]:
facts = set()
for _, r in corpus.iterrows():
    facts.add(("тип", r["id"], r["тип"]))
    for d in filter(None, r["зависит_от"].split(";")):
        facts.add(("зависит_от", r["id"], d))
for _, r in reviews.iterrows():
    rid = f"рец_{r['статья']}_{r['рецензент']}"
    facts |= {("тип", rid, "Рецензия"), ("на_статью", rid, r["статья"]), ("рецензент", rid, r["рецензент"]),
              ("из_организации", r["рецензент"], r["рецензент_организация"]),
              ("организация_авторов", r["статья"], r["авторы_организация"]),
              ("тип", r["рецензент_организация"], "Организация"), ("тип", r["авторы_организация"], "Организация")}

RULES = [
    ("транзитивность", [("зависит_от", "?x", "?y"), ("зависит_от", "?y", "?z")], ("зависит_от", "?x", "?z")),
    ("требование 6.2", [("тип", "?r", "Рецензия"), ("на_статью", "?r", "?s"), ("рецензент", "?r", "?p"),
                        ("из_организации", "?p", "?o"), ("организация_авторов", "?s", "?o")], ("нарушает", "?s", "R62")),
    ("область значений", [("рецензент", "?r", "?p")], ("тип", "?p", "Рецензент")),
]

def match(premises, facts, env=None):
    """Все подстановки переменных, при которых условия правила выполнены на фактах."""
    env = env or {}
    if not premises:
        yield env; return
    first, rest = premises[0], premises[1:]
    for f in facts:
        e = dict(env)
        if all((t.startswith("?") and e.setdefault(t, v) == v) or (not t.startswith("?") and t == v) for t, v in zip(first, f)):
            yield from match(rest, facts, e)

def closure(facts, rules):
    facts, step = set(facts), 0
    while True:
        new = {tuple(env.get(t, t) for t in head) for _, body, head in rules for env in match(body, facts)} - facts
        step += 1
        print(f"шаг {step}: новых фактов {len(new)}")
        if not new:
            return facts
        facts |= new

closed = closure(facts, RULES)
print("R65 зависит_от:", len({o for p, s, o in closed if p == "зависит_от" and s == "R65"}), "положений")
print("нарушают R62:", sorted(s for p, s, o in closed if p == "нарушает"))

шаг 1: новых фактов 73
шаг 2: новых фактов 19
шаг 3: новых фактов 0
R65 зависит_от: 14 положений
нарушают R62: ['P3']


Собственный прямой вывод дал те же 14 положений и то же нарушение. Замыкание сошлось за три шага:
на третьем шаге новых фактов нет, это и есть неподвижная точка.

### 7б. Вывод как операция над матрицей

Отношение «подкласс» записывается булевой матрицей S. Транзитивное замыкание S* = S ∨ S² ∨ … вычисляется
до неподвижной точки. Булево произведение матриц соответствует композиции отношений.

In [11]:
CLS = ["Требование", "Термин", "Раздел", "Положение", "Рецензент", "Организация", "Объект"]
ci = {c: i for i, c in enumerate(CLS)}
S = np.zeros((len(CLS), len(CLS)), dtype=bool)
for a, b in [("Требование", "Положение"), ("Термин", "Положение"), ("Раздел", "Положение"),
             ("Положение", "Объект"), ("Рецензент", "Объект"), ("Организация", "Объект")]:
    S[ci[a], ci[b]] = True

def star(S):
    R = S.copy()
    while True:
        R2 = R | ((R.astype(int) @ S.astype(int)) > 0)      # булево произведение = композиция
        if (R2 == R).all():
            return R
        R = R2

Sstar = star(S)
show = lambda A, rows, cols: pd.DataFrame(A.astype(int), index=rows, columns=cols)
print("S: строка ⊑ столбец"); display(show(S, CLS, CLS))
print("S*: добавилось Требование ⊑ Объект, Термин ⊑ Объект, Раздел ⊑ Объект"); display(show(Sstar, CLS, CLS))

S: строка ⊑ столбец


,Требование,Термин,Раздел,Положение,Рецензент,Организация,Объект
Требование,0,0,0,1,0,0,0
Термин,0,0,0,1,0,0,0
Раздел,0,0,0,1,0,0,0
Положение,0,0,0,0,0,0,1
Рецензент,0,0,0,0,0,0,1
Организация,0,0,0,0,0,0,1
Объект,0,0,0,0,0,0,0


S*: добавилось Требование ⊑ Объект, Термин ⊑ Объект, Раздел ⊑ Объект


,Требование,Термин,Раздел,Положение,Рецензент,Организация,Объект
Требование,0,0,0,1,0,0,1
Термин,0,0,0,1,0,0,1
Раздел,0,0,0,1,0,0,1
Положение,0,0,0,0,0,0,1
Рецензент,0,0,0,0,0,0,1
Организация,0,0,0,0,0,0,1
Объект,0,0,0,0,0,0,0


### 7в. Распространение классов

Матрица M задаёт принадлежность индивидов классам, как записано. Принадлежность надклассам:
M′ = max(M, M·S*).

In [12]:
def membership(individuals, facts):
    M = np.zeros((len(individuals), len(CLS)), dtype=bool)
    for i, x in enumerate(individuals):
        for p, s, o in facts:
            if p == "тип" and s == x and o in ci:
                M[i, ci[o]] = True
    return M

def propagate(M):
    return M | ((M.astype(int) @ Sstar.astype(int)) > 0)

IND = ["R65", "T06", "S01", "Rv4", "ФИЦ ИУ"]
M = membership(IND, closed)
print("M: записано и получено по области значений"); display(show(M, IND, CLS))
M1 = propagate(M)
print("M′ = max(M, M·S*)"); display(show(M1, IND, CLS))

M: записано и получено по области значений


,Требование,Термин,Раздел,Положение,Рецензент,Организация,Объект
R65,1,0,0,0,0,0,0
T06,0,1,0,0,0,0,0
S01,0,0,1,0,0,0,0
Rv4,0,0,0,0,1,0,0
ФИЦ ИУ,0,0,0,0,0,1,0


M′ = max(M, M·S*)


,Требование,Термин,Раздел,Положение,Рецензент,Организация,Объект
R65,1,0,0,1,0,0,1
T06,0,1,0,1,0,0,1
S01,0,0,1,1,0,0,1
Rv4,0,0,0,0,1,0,1
ФИЦ ИУ,0,0,0,0,0,1,1


## Шаг 8. Строка журнала с ошибкой

В журнал добавляется пятнадцатая строка. В поле «рецензент» по ошибке записана организация «ФИЦ ИУ».

### 8а. Проверка непротиворечивости как операция над матрицей

Дизъюнктность классов задаётся матрицей D. Условие (M′·D) ⊙ M′ > 0 означает, что найден индивид,
принадлежащий двум непересекающимся классам.

In [13]:
D = np.zeros((len(CLS), len(CLS)), dtype=bool)
for a, b in [("Рецензент", "Организация"), ("Рецензент", "Положение"), ("Организация", "Положение"),
             ("Требование", "Термин"), ("Требование", "Раздел"), ("Термин", "Раздел")]:
    D[ci[a], ci[b]] = D[ci[b], ci[a]] = True

def conflicts(individuals, facts):
    M1 = propagate(membership(individuals, facts))
    bad = ((M1.astype(int) @ D.astype(int)) > 0) & M1
    return [(individuals[i], [CLS[j] for j in np.where(bad[i])[0]]) for i in range(len(individuals)) if bad[i].any()]

print("до ошибки:", conflicts(IND, closed) or "противоречий нет")

bad_facts = {("тип", "рец_P9_ошибка", "Рецензия"), ("на_статью", "рец_P9_ошибка", "P8"), ("рецензент", "рец_P9_ошибка", "ФИЦ ИУ")}
closed_bad = closure(facts | bad_facts, RULES)
print("после ошибки:", conflicts(IND, closed_bad))

до ошибки: противоречий нет
шаг 1: новых фактов 74
шаг 2: новых фактов 19
шаг 3: новых фактов 0
после ошибки: [('ФИЦ ИУ', ['Рецензент', 'Организация'])]


### 8б. То же в машине вывода

Pellet сообщает о противоречии и называет участвующие аксиомы и факты.

In [14]:
RECORD_8 = '''[запись] сообщение машины вывода без Java
Explanation(s):
1)   рецензент range Рецензент
     ФИЦ_ИУ type Организация
     рец_P9_ошибка рецензент ФИЦ_ИУ
     DisjointClasses: Положение, Статья, Рецензия, Рецензент, Организация, Утверждение, Доказательство'''
if HAS_JAVA:
    with onto:
        ошибка = Рецензия("рец_P9_ошибка")
        ошибка.на_статью = art["P8"]
        ошибка.рецензент = org["ФИЦ ИУ"]
    ok, msg = run_reasoner(explain=True)
    print("непротиворечива:", ok)
    print(re.sub(r"\n\s*\n", "\n", msg.replace("review.", "")).strip()[:700])
    destroy_entity(ошибка)
    ok, msg = run_reasoner()
    print("\nпосле исправления строки, непротиворечива:", ok)
else:
    print(RECORD_8)

непротиворечива: False
Explanation(s): 
1)   DisjointClasses(Доказательство
                     Организация
                     Положение
                     Рецензент
                     Рецензия
                     Статья
                     Утверждение)
     рец_P9_ошибка рецензент ФИЦ_ИУ
     рецензент range Рецензент
     ФИЦ_ИУ type Организация

после исправления строки, непротиворечива: True


## Шаг 9. Чего в выводе нет

Машина вывода нашла нарушение требования 6.2. Требования 6.1 (не меньше двух сданных рецензий)
и 6.6 (рецензия в 21 день) она не проверила.

In [15]:
rows = []
for name, a in sorted(art.items()):
    sub = reviews[(reviews["статья"] == name) & (reviews["сдано"] != "")]
    days = [(day(r["сдано"]) - day(r["назначено"])).days for _, r in sub.iterrows()]
    rows.append({"статья": name, "нарушает (вывод машины)": ", ".join(x.name for x in a.нарушает) or "—",
                 "сдано рецензий": len(sub), "макс. дней": max(days) if days else None})
display(pd.DataFrame(rows))
print("P4 и P8: одна или ноль сданных рецензий. P5: 27 дней при пороге 21. Машина вывода об этом молчит.")

,статья,нарушает (вывод машины),сдано рецензий,макс. дней
0,P1,—,2,17.0
1,P2,—,2,19.0
2,P3,R62,2,17.0
3,P4,—,1,12.0
4,P5,—,2,27.0
5,P6,—,2,13.0
6,P7,—,2,16.0
7,P8,—,0,NaN


P4 и P8: одна или ноль сданных рецензий. P5: 27 дней при пороге 21. Машина вывода об этом молчит.


Причина в открытом мире. Отсутствие записи о второй рецензии не означает, что её нет. Сравнение дат
относится к проверке данных. Обе проверки выполняются на замкнутом мире средствами SHACL, это лекция 3.

| Механизм | Что дал |
|---|---|
| транзитивное свойство | 14 положений, от которых зависит требование 6.5 |
| правило Хорна (SWRL) | статья P3 нарушает требование 6.2 |
| иерархия классов | R65 является положением и объектом |
| определяемый класс и замыкание | разделы 1 и 2 базовые |
| область значений и дизъюнктность | строка журнала с ошибкой обнаружена как противоречие |
| проверка на замкнутом мире | требования 6.1 и 6.6, лекция 3 |

## Шаг 10. Агент с онтологией

Мини-агент лекции 1 получает два новых инструмента. `infer` возвращает факты, выведенные машиной вывода,
с указанием правила. `check_consistency` возвращает вердикт о непротиворечивости записи.
Правило 6.2 теперь находится в онтологии, в коде инструмента его нет.

Ячейка работает с моделью через OpenAI-совместимый API (`LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`), настроенной
в ячейке «Модель» в начале блокнота. Если модель не настроена или сервер не отвечает, печатается протокол: вызовы инструментов выполняются на самом деле, текст ответа модели записан заранее.

In [16]:
def infer(article: str):
    a = art.get(article)
    if a is None:
        return {"ошибка": f"статья {article} не найдена"}
    return {"статья": article,
            "выведено": [{"факт": f"{article} нарушает {x.name} ({x.label[0]})", "правило": "правило 6.2 (SWRL)"} for x in a.нарушает],
            "организация_авторов": [o.name for o in a.организация_авторов]}

def graph_deps(doc_id: str):
    x = pos.get(doc_id)
    return {"положение": doc_id, "зависит_от": sorted(f"{d.name} {d.label[0]}" for d in x.зависит_от), "основание": "транзитивность зависит_от"}

def check_consistency():
    if not HAS_JAVA:
        bad = conflicts(IND, closure_quiet(facts | (bad_facts if AGENT_ERROR_ROW else set())))
        return {"непротиворечива": not bad, "конфликт": bad}
    ok, msg = run_reasoner(explain=True)
    return {"непротиворечива": ok, "объяснение": re.sub(r"\s+", " ", msg.replace("review.", ""))[:300]}

def closure_quiet(f):
    with contextlib.redirect_stdout(io.StringIO()):
        return closure(f, RULES)

TOOLS = [
  {"type": "function", "function": {"name": "infer", "description": "Факты о статье, выведенные машиной вывода по онтологии, с указанием правила.",
     "parameters": {"type": "object", "properties": {"article": {"type": "string", "description": "например P3"}}, "required": ["article"]}}},
  {"type": "function", "function": {"name": "graph_deps", "description": "От каких положений зависит данное положение (с учётом транзитивности).",
     "parameters": {"type": "object", "properties": {"doc_id": {"type": "string"}}, "required": ["doc_id"]}}},
  {"type": "function", "function": {"name": "check_consistency", "description": "Проверка непротиворечивости записи журнала и положения.",
     "parameters": {"type": "object", "properties": {}}}},
]
IMPL = {"infer": infer, "graph_deps": graph_deps, "check_consistency": check_consistency}
SYSTEM = ("Ты ассистент редактора сборника. Отвечай по-русски. Сначала вызови check_consistency. Если запись противоречива, сообщи об этом "
          "с пометкой [противоречие в данных] и не делай выводов о статьях. Иначе отвечай только по результатам инструментов. "
          "Помечай части ответа: [выведено: правило], [найдено: инструмент], [не подтверждено].")
QUESTION = "Нарушает ли статья P3 требования положения о рецензировании, и на какие положения опирается нарушенное требование?"

def run_agent(question, max_steps=8):
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ.get("LLM_BASE_URL"))
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        m = client.chat.completions.create(model=os.environ.get("LLM_MODEL", "gpt-4o-mini"), messages=messages, tools=TOOLS).choices[0].message
        messages.append(m)
        if not m.tool_calls:
            print(f"[{step}] ответ модели:\n{m.content}"); return
        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            text = json.dumps(IMPL[tc.function.name](**args), ensure_ascii=False)
            print(f"[{step}] модель вызвала {tc.function.name}({args})\n    ← {text[:170]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": text})

def protocol(calls, answer):
    """Протокол без модели: инструменты вызываются на самом деле, текст ответа записан заранее."""
    print("[запись ответа модели, вызовы инструментов настоящие]")
    for i, (name, args) in enumerate(calls, 1):
        text = json.dumps(IMPL[name](**args), ensure_ascii=False)
        print(f"[{i}] модель вызвала {name}({args})\n    ← {text[:170]}{'…' if len(text) > 170 else ''}")
    print(f"[{len(calls) + 1}] ответ модели:\n{answer}")

AGENT_ERROR_ROW = False
print("=== Прогон 1: вывод ===")
if os.environ.get("LLM_API_KEY"):
    run_agent(QUESTION)
else:
    protocol([("check_consistency", {}), ("infer", {"article": "P3"}), ("graph_deps", {"doc_id": "R62"})],
             "Запись непротиворечива [найдено: check_consistency]. Статья P3 нарушает требование 6.2 «Независимость рецензента» "
             "[выведено: правило 6.2]. Требование 6.2 опирается на раздел 6, раздел 3, раздел 1 и термины Рецензент, Автор, "
             "Статья, Редакционная коллегия [найдено: graph_deps]. О требованиях 6.1 и 6.6 инструменты сведений не дали [не подтверждено].")

=== Прогон 1: вывод ===
[запись ответа модели, вызовы инструментов настоящие]
[1] модель вызвала check_consistency({})
    ← {"непротиворечива": true, "объяснение": ""}
[2] модель вызвала infer({'article': 'P3'})
    ← {"статья": "P3", "выведено": [{"факт": "P3 нарушает R62 (6.2 Независимость рецензента)", "правило": "правило 6.2 (SWRL)"}], "организация_авторов": ["ФИЦ_ИУ"]}
[3] модель вызвала graph_deps({'doc_id': 'R62'})
    ← {"положение": "R62", "зависит_от": ["S01 Раздел 1. Область применения", "S03 Раздел 3. Термины и определения", "S06 Раздел 6. Порядок рецензирования", "T01 Статья", "T04 …
[4] ответ модели:
Запись непротиворечива [найдено: check_consistency]. Статья P3 нарушает требование 6.2 «Независимость рецензента» [выведено: правило 6.2]. Требование 6.2 опирается на раздел 6, раздел 3, раздел 1 и термины Рецензент, Автор, Статья, Редакционная коллегия [найдено: graph_deps]. О требованиях 6.1 и 6.6 инструменты сведений не дали [не подтверждено].


In [17]:
print("=== Прогон 2: журнал со строкой с ошибкой ===")
AGENT_ERROR_ROW = True
if HAS_JAVA:
    with onto:
        ошибка = Рецензия("рец_P9_ошибка"); ошибка.на_статью = art["P8"]; ошибка.рецензент = org["ФИЦ ИУ"]
if os.environ.get("LLM_API_KEY"):
    run_agent(QUESTION)
else:
    protocol([("check_consistency", {})],
             "Запись журнала противоречива [противоречие в данных]: индивид ФИЦ_ИУ указан рецензентом в строке рец_P9_ошибка "
             "и одновременно является организацией. Выводы о статьях не делаю до исправления строки.")
if HAS_JAVA:
    destroy_entity(ошибка); run_reasoner()
AGENT_ERROR_ROW = False

=== Прогон 2: журнал со строкой с ошибкой ===
[запись ответа модели, вызовы инструментов настоящие]
[1] модель вызвала check_consistency({})
    ← {"непротиворечива": false, "объяснение": "Explanation(s): 1) DisjointClasses(Доказательство Организация Положение Рецензент Рецензия Статья Утверждение) рец_P9_ошибка рец…
[2] ответ модели:
Запись журнала противоречива [противоречие в данных]: индивид ФИЦ_ИУ указан рецензентом в строке рец_P9_ошибка и одновременно является организацией. Выводы о статьях не делаю до исправления строки.


Смотреть на три вещи. Правила 6.2 в коде инструментов нет, оно находится в онтологии. Каждый факт ответа помечен
источником или правилом. На испорченных данных агент останавливается: проверка непротиворечивости возвращает
рассогласование, и вывод по статьям откладывается до исправления.

## Шаг 11. Вариант по математике

Тот же скелет на корпусе планиметрии из лекции 1: утверждения, зависимости между ними и доказательства.
Иерархия Лемма ⊑ Утверждение ⊑ Объект взята из презентации курса.

In [18]:
plan = pd.read_csv(DATA / "corpus_A_planimetry.csv", dtype=str).fillna("")
mfacts = {("зависит_от", r["id"], d) for _, r in plan.iterrows() for d in filter(None, r["зависит_от"].split(";"))}
for _, c in claims.iterrows():
    mfacts |= {("зависит_от", c["id"], d) for d in filter(None, c["опирается_на"].split(";"))}
mclosed = closure(mfacts, RULES[:1])
title = dict(zip(plan["id"], plan["название"]))
for start in ("A14", "C1"):
    deps = sorted(o for p, s, o in mclosed if s == start)
    print(f"{start} зависит от {len(deps)} утверждений:", ", ".join(deps))

MC = ["Лемма", "Теорема", "Утверждение", "Объект"]
Sm = np.zeros((4, 4), dtype=bool); Sm[0, 2] = Sm[1, 2] = Sm[2, 3] = True
R = Sm | ((Sm.astype(int) @ Sm.astype(int)) > 0)
print("\nS* для иерархии Лемма ⊑ Утверждение ⊑ Объект:"); display(pd.DataFrame(R.astype(int), index=MC, columns=MC))
display(claims[["id", "вид", "опирается_на", "вердикт_lean"]])

шаг 1: новых фактов 63
шаг 2: новых фактов 57
шаг 3: новых фактов 8
шаг 4: новых фактов 0
A14 зависит от 11 утверждений: A01, A02, A03, A04, A05, A06, A07, A10, A11, A12, A13
C1 зависит от 13 утверждений: A01, A02, A03, A04, A05, A06, A07, A10, A11, A12, A13, A14, A27

S* для иерархии Лемма ⊑ Утверждение ⊑ Объект:


,Лемма,Теорема,Утверждение,Объект
Лемма,0,0,1,1
Теорема,0,0,1,1
Утверждение,0,0,0,1
Объект,0,0,0,0


,id,вид,опирается_на,вердикт_lean
0,C1,теорема,A05;A14;A27,принято
1,C2,лемма,A05;A13;A20,принято
2,C3,утверждение,A27,отклонено: доказательство не собирается; контр...


Теорема Пифагора зависит от 11 утверждений, как в лекции 1. Утверждения статьи P2 встают в тот же граф:
их зависимости продолжаются в корпус планиметрии. Вердикт внешней проверки хранится как значение свойства
у доказательства.

## Что повторить дома

1. Запустить блокнот целиком. Открыть `data/review.owl` в Protégé.
2. Записать фрагмент своей области по образцу шагов 1–4 и сохранить в `kg/ontology.ttl`.
3. Запустить машину вывода, выписать один выведенный факт.
4. Внести одну намеренную ошибку в данные, получить сообщение о противоречии, исправить.
5. Записать результат в `kg/ONTOLOGY.md` и в `diary.md`.